In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
import statsmodels.api as sm
from scipy import stats
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

df = pd.read_csv("Auto-1.csv")

df = df.drop(columns=["name"])

y = df["mpg"]
X = df.drop(columns=["mpg"])

X_const = sm.add_constant(X)

In [2]:
X1 = sm.add_constant(X)

X2 = sm.add_constant(X.drop(columns=["cylinders"]))

poly = PolynomialFeatures(degree=2, include_bias=False)

X3_poly = poly.fit_transform(X)

X3 = pd.DataFrame(
    X3_poly,
    columns=poly.get_feature_names_out(X.columns),
    index=X.index
)

X3 = sm.add_constant(X3)

X4 = X.copy()

X4["log_weight"] = np.log(X4["weight"])

X4 = X4.drop(columns=["weight"])

X4 = sm.add_constant(X4)

poly2 = PolynomialFeatures(degree=2, include_bias=False)

X5_poly = poly2.fit_transform(X[["weight", "horsepower"]])

X5 = pd.DataFrame(
    X5_poly,
    columns=poly2.get_feature_names_out(["weight", "horsepower"]),
    index=X.index
)

X5 = sm.add_constant(X5)

models_X = [X1, X2, X3, X4, X5]

model_names = [
    "Model 1: Linear",
    "Model 2: Remove Cylinders",
    "Model 3: Polynomial All",
    "Model 4: Log Weight",
    "Model 5: Polynomial Weight & HP"]

In [3]:
results_table = []

for i, X_model in enumerate(models_X):
    model = sm.OLS(y, X_model).fit()
    results_table.append([
        model_names[i],
        model.rsquared_adj,
        model.aic,
        model.bic
    ])

results_df = pd.DataFrame(
    results_table,
    columns=["Model", "Adjusted R2", "AIC", "BIC"]
)

print("\nModel Comparison (Adjusted R2, AIC, BIC)")
print(results_df)


Model Comparison (Adjusted R2, AIC, BIC)
                             Model  Adjusted R2          AIC          BIC
0                  Model 1: Linear     0.703906  2252.242529  2276.070100
1        Model 2: Remove Cylinders     0.703953  2251.195457  2271.051767
2          Model 3: Polynomial All     0.761042  2182.665868  2266.062367
3              Model 4: Log Weight     0.724490  2223.997896  2247.825467
4  Model 5: Polynomial Weight & HP     0.746184  2191.848214  2215.675785


In [4]:
n_reps = 100
mse_results = np.zeros((5, n_reps))

for i in range(n_reps):

    train_index, test_index = train_test_split(
        np.arange(len(df)),
        test_size=0.2
    )

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    for j, X_model in enumerate(models_X):

        X_train = X_model.iloc[train_index]
        X_test = X_model.iloc[test_index]

        model = sm.OLS(y_train, X_train).fit()

        y_pred = model.predict(X_test)

        mse = mean_squared_error(y_test, y_pred)

        mse_results[j, i] = mse

avg_mse = mse_results.mean(axis=1)

validation_df = pd.DataFrame({
    "Model": model_names,
    "Average Validation MSE": avg_mse
})

print("\nAverage Validation MSE (100 repetitions)")
print(validation_df)


Average Validation MSE (100 repetitions)
                             Model  Average Validation MSE
0                  Model 1: Linear               18.451810
1        Model 2: Remove Cylinders               18.414471
2          Model 3: Polynomial All               16.253428
3              Model 4: Log Weight               17.096596
4  Model 5: Polynomial Weight & HP               15.443009


In [5]:
kf = KFold(n_splits=10, shuffle=True, random_state=1)

cv_mse = np.zeros(5)

for j in range(5):

    mse_list = []

    for train_index, test_index in kf.split(models_X[j]):

        X_train = models_X[j].iloc[train_index]
        X_test = models_X[j].iloc[test_index]

        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]

        model = sm.OLS(y_train, X_train).fit()

        y_pred = model.predict(X_test)

        mse = mean_squared_error(y_test, y_pred)

        mse_list.append(mse)

    cv_mse[j] = np.mean(mse_list)

cv_df = pd.DataFrame({
    "Model": model_names,
    "10-Fold CV MSE": cv_mse
})

print(cv_df)


                             Model  10-Fold CV MSE
0                  Model 1: Linear       18.382582
1        Model 2: Remove Cylinders       18.312905
2          Model 3: Polynomial All       16.463644
3              Model 4: Log Weight       17.117429
4  Model 5: Polynomial Weight & HP       15.619612


In [7]:
final_df = results_df.copy()

final_df["Validation MSE"] = avg_mse

final_df["CV MSE"] = cv_mse

print(final_df)

                             Model  Adjusted R2          AIC          BIC  \
0                  Model 1: Linear     0.703906  2252.242529  2276.070100   
1        Model 2: Remove Cylinders     0.703953  2251.195457  2271.051767   
2          Model 3: Polynomial All     0.761042  2182.665868  2266.062367   
3              Model 4: Log Weight     0.724490  2223.997896  2247.825467   
4  Model 5: Polynomial Weight & HP     0.746184  2191.848214  2215.675785   

   Validation MSE     CV MSE  
0       18.451810  18.382582  
1       18.414471  18.312905  
2       16.253428  16.463644  
3       17.096596  17.117429  
4       15.443009  15.619612  


In [8]:
best_adj_r2 = final_df.loc[final_df["Adjusted R2"].idxmax(), "Model"]

best_validation = final_df.loc[final_df["Validation MSE"].idxmin(), "Model"]

best_cv = final_df.loc[final_df["CV MSE"].idxmin(), "Model"]

print("Best Adjusted R2:", best_adj_r2)
print("Best Validation MSE:", best_validation)
print("Best CV MSE:", best_cv)

Best Adjusted R2: Model 3: Polynomial All
Best Validation MSE: Model 5: Polynomial Weight & HP
Best CV MSE: Model 5: Polynomial Weight & HP
